# DSC602 — Information Visualization (Homework 1)

**Dataset:** Old cars (1970–1982)  
**Objectives:** basic visualization techniques + Matplotlib familiarity

This notebook:
- Loads and cleans the dataset
- Performs basic exploratory data analysis (EDA)
- Implements **Task 1–Task 4** exactly as stated in the assignment


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pandas.plotting import scatter_matrix

plt.style.use("seaborn-v0_8")
%matplotlib inline

## 1) Load data

In [ ]:
# Try common local paths first
CANDIDATE_PATHS = [
    "old_cars - old_cars.csv",
    os.path.join(".", "old_cars - old_cars.csv"),
    os.path.join("assignment", "DSC602", "old_cars - old_cars.csv"),
]

csv_path = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError(
        "Could not find 'old_cars - old_cars.csv'. "
        "Place the CSV next to this notebook or update CANDIDATE_PATHS."
    )

df = pd.read_csv(csv_path)
csv_path, df.shape

### Quick preview

In [ ]:
df.head(10)

## 2) Data cleaning + typing

The dataset should have columns:
- `Car` (string)
- `MPG`, `Displacement`, `Horsepower`, `Weight` (numeric)
- `Model` (year, numeric)
- `Origin` (categorical: US / Europe / Japan)

Common issue in this dataset: some `Horsepower` values are **0** (often used as missing). We treat `Horsepower == 0` as missing.

In [ ]:
df = df.copy()

# Standardize column names (just in case)
df.columns = [c.strip() for c in df.columns]

# Convert numeric columns
numeric_cols = ["MPG", "Displacement", "Horsepower", "Weight", "Model"]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Treat 0 horsepower as missing (dataset convention)
df.loc[df["Horsepower"] == 0, "Horsepower"] = np.nan

# Clean up categories
df["Origin"] = df["Origin"].astype(str).str.strip()
df["Car"] = df["Car"].astype(str).str.strip()

df.info()

### Missing values

In [ ]:
df.isna().sum().sort_values(ascending=False)

### Summary statistics

In [ ]:
df.describe(include="all").T

### Distribution sanity checks

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

cols = ["MPG", "Displacement", "Horsepower", "Weight"]
for ax, c in zip(axes, cols):
    ax.hist(df[c].dropna(), bins=20, color="#4C72B0", edgecolor="white")
    ax.set_title(f"Histogram: {c}")
    ax.set_xlabel(c)
    ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

## Task 1: Grouped bar chart MPG distribution by Origin (2 MPG increments)

**Requirement:** Discretize MPG values into **2-MPG** bins. For each MPG bin, show **3 bars** (US, Europe, Japan) so distributions compare across origins.

**Approach:**
- Build bins from `min(MPG)` to `max(MPG)` in steps of 2
- Use `pd.cut` to assign each car to a bin
- Count cars per bin per origin
- Plot grouped bars with offsets

In [ ]:
mpg = df["MPG"].dropna()

# Build 2-MPG increment bins
min_mpg = np.floor(mpg.min() / 2) * 2
max_mpg = np.ceil(mpg.max() / 2) * 2
bins = np.arange(min_mpg, max_mpg + 2, 2)

df_task1 = df.dropna(subset=["MPG", "Origin"]).copy()
df_task1["MPG_bin"] = pd.cut(df_task1["MPG"], bins=bins, right=False)

# Count per bin and origin
counts = (
    df_task1.groupby(["MPG_bin", "Origin"])  
    .size()
    .unstack(fill_value=0)
)

# Ensure consistent origin order
origin_order = [o for o in ["US", "Europe", "Japan"] if o in counts.columns]
counts = counts.reindex(columns=origin_order)

counts.head()

In [ ]:
x = np.arange(len(counts.index))
bar_width = 0.25

fig, ax = plt.subplots(figsize=(16, 6))

colors = {"US": "#4C72B0", "Europe": "#55A868", "Japan": "#C44E52"}

for i, origin in enumerate(origin_order):
    ax.bar(
        x + (i - (len(origin_order) - 1) / 2) * bar_width,
        counts[origin].values,
        width=bar_width,
        label=origin,
        color=colors.get(origin, None),
        edgecolor="white",
    )

ax.set_title("Task 1 — MPG distribution (2-MPG bins) by Origin")
ax.set_xlabel("MPG bin (2-MPG increments)")
ax.set_ylabel("Number of cars")

# Make readable bin labels
bin_labels = [f"{int(iv.left)}–{int(iv.left+2)}" for iv in counts.index]
ax.set_xticks(x)
ax.set_xticklabels(bin_labels, rotation=45, ha="right")

ax.legend(title="Origin")
plt.tight_layout()
plt.show()

## Task 2: Line chart evolution of average MPG (1970–1982) by Origin

**Requirement:** For each origin, plot **annual average MPG** from **1970 to 1982** (13 points per curve), with different colors.

In [ ]:
df_task2 = df.dropna(subset=["MPG", "Model", "Origin"]).copy()
df_task2 = df_task2[(df_task2["Model"] >= 70) & (df_task2["Model"] <= 82)]

avg_mpg = (
    df_task2.groupby(["Model", "Origin"])  
    ["MPG"].mean()
    .unstack()
)

# Ensure all years 70..82 are present (even if missing for an origin)
years = np.arange(70, 83)
avg_mpg = avg_mpg.reindex(years)
avg_mpg = avg_mpg.reindex(columns=origin_order)

avg_mpg.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for origin in origin_order:
    ax.plot(
        years,
        avg_mpg[origin],
        marker="o",
        linewidth=2,
        label=origin,
        color=colors.get(origin, None),
    )

ax.set_title("Task 2 — Annual average MPG by Origin (1970–1982)")
ax.set_xlabel("Model year")
ax.set_ylabel("Average MPG")
ax.set_xticks(years)
ax.legend(title="Origin")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Task 3: Scatter plot Horsepower vs MPG (color = year)

**Requirement:**
- x-axis: `Horsepower`
- y-axis: `MPG`
- each point: a car
- color coding: `Model` year

In [ ]:
df_task3 = df.dropna(subset=["Horsepower", "MPG", "Model"]).copy()

fig, ax = plt.subplots(figsize=(10, 6))

sc = ax.scatter(
    df_task3["Horsepower"],
    df_task3["MPG"],
    c=df_task3["Model"],
    cmap="viridis",
    alpha=0.75,
    edgecolor="white",
    linewidth=0.4,
)

ax.set_title("Task 3 — Horsepower vs MPG (color = model year)")
ax.set_xlabel("Horsepower")
ax.set_ylabel("MPG")
ax.grid(True, alpha=0.25)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Model year")

plt.tight_layout()
plt.show()

## Task 4: Scatter plot matrix MPG, Weight, Horsepower, Displacement (color = Origin)

**Requirement:** Use a scatter plot matrix for:
- `MPG`, `Weight`, `Horsepower`, `Displacement`

Color code data points by `Origin`.



In [ ]:
cols_matrix = ["MPG", "Weight", "Horsepower", "Displacement"]
df_task4 = df.dropna(subset=cols_matrix + ["Origin"]).copy()

# Map origins to colors for scatter_matrix
origin_to_color = {"US": "#4C72B0", "Europe": "#55A868", "Japan": "#C44E52"}
point_colors = df_task4["Origin"].map(origin_to_color).fillna("#777777")

axes = scatter_matrix(
    df_task4[cols_matrix],
    figsize=(12, 12),
    diagonal="hist",
    alpha=0.7,
    c=point_colors,
    marker="o",
    s=25,
    range_padding=0.1,
)

# Add a custom legend (scatter_matrix doesn't add it automatically)
import matplotlib.patches as mpatches

legend_handles = [
    mpatches.Patch(color=origin_to_color[o], label=o)
    for o in origin_order
    if o in df_task4["Origin"].unique()
]

plt.suptitle("Task 4 — Scatter plot matrix (color = Origin)", y=1.02)
plt.legend(handles=legend_handles, title="Origin", loc="upper right", bbox_to_anchor=(1.15, 1.02))

plt.tight_layout()
plt.show()